# Pollution Control Simulations — ECON7720 Lecture 04

Four interactive simulations that follow the lecture structure:

1. **Command-and-control** — the excess cost of a uniform standard
2. **Price instrument (tax)** — how a tax achieves cost-effectiveness
3. **Quantity instrument (cap-and-trade)** — permit market simulation
4. **Weitzman (1974)** — prices vs quantities under uncertainty

---
*ECON7720 — Ecological & Environmental Economics | The University of Queensland | Dr Juan Soto-Diaz*

In [ ]:
%pip install -q ipywidgets matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

# UQ branding
UQ_PURPLE = "#512478"
UQ_CYAN = "#0099CC"
COLORS = [UQ_PURPLE, UQ_CYAN, "#E6308A", "#2EA836", "#E87722", "#00A4BD",
          "#8B5CF6", "#DC2626"]

E0_PER_FIRM = 10.0  # uncontrolled emissions per firm

---
## 1. Command-and-control — the excess cost of a uniform standard

Two firms with different MAC slopes share a total abatement target.

- The **equimarginal split** (where MAC₁ = MAC₂) minimises total cost.
- A **uniform standard** forces equal abatement — the shaded triangle is the **excess cost**.

Use the sliders to see how heterogeneity and the abatement target affect the waste.

In [ ]:
def plot_uniform_standard(total_abatement=10, slope1=3.0, slope2=1.0):
    """Excess cost of a uniform standard vs equimarginal allocation."""
    # Equimarginal: slope1*a1 = slope2*a2, a1+a2 = A
    # => a1 = A * slope2/(slope1+slope2), a2 = A * slope1/(slope1+slope2)
    A = total_abatement
    a1_star = A * slope2 / (slope1 + slope2)
    a2_star = A * slope1 / (slope1 + slope2)
    p_star = slope1 * a1_star  # = slope2 * a2_star

    a_uniform = A / 2

    # Costs
    cost_eq = 0.5 * slope1 * a1_star**2 + 0.5 * slope2 * a2_star**2
    cost_un = 0.5 * slope1 * a_uniform**2 + 0.5 * slope2 * a_uniform**2
    excess = cost_un - cost_eq

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Command-and-Control: Excess Cost of a Uniform Standard",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    # -- Left panel: back-to-back MAC diagram --
    ax = axes[0]
    ax.set_title("Abatement allocation", fontsize=11, color=UQ_PURPLE)

    a_max = max(A * 1.1, 2)
    a_range = np.linspace(0, a_max, 300)

    # MAC1 from left
    ax.plot(a_range, slope1 * a_range, color=UQ_PURPLE, lw=2, label=f"$MAC_1$ (slope={slope1:.1f})")
    # MAC2 from right (abatement increases right-to-left)
    ax.plot(A - a_range, slope2 * a_range, color=UQ_CYAN, lw=2, label=f"$MAC_2$ (slope={slope2:.1f})")

    # Equimarginal point
    ax.plot(a1_star, p_star, "o", color=UQ_PURPLE, ms=8, zorder=5)
    ax.axvline(a1_star, color="gray", ls="--", lw=1, alpha=0.5)
    ax.annotate(f"$a_1^*={a1_star:.1f}$", xy=(a1_star, 0),
                xytext=(a1_star, -0.08 * ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else -0.5),
                fontsize=9, ha="center", color=UQ_PURPLE)

    # Uniform line
    ax.axvline(a_uniform, color="gray", ls=":", lw=2, alpha=0.7)
    ax.text(a_uniform, slope1 * a_uniform * 1.05, "uniform", fontsize=8,
            ha="center", color="gray")

    # Excess cost triangle
    if a_uniform > a1_star:
        a_tri = np.linspace(a1_star, a_uniform, 100)
        mac1_tri = slope1 * a_tri
        mac2_tri = slope2 * (A - a_tri)
        ax.fill_between(a_tri, mac2_tri, mac1_tri, alpha=0.25, color=UQ_PURPLE,
                        label=f"Excess cost = ${excess:.1f}")
    else:
        a_tri = np.linspace(a_uniform, a1_star, 100)
        mac1_tri = slope1 * a_tri
        mac2_tri = slope2 * (A - a_tri)
        ax.fill_between(a_tri, mac1_tri, mac2_tri, alpha=0.25, color=UQ_PURPLE,
                        label=f"Excess cost = ${excess:.1f}")

    ax.set_xlabel(f"Firm 1 abatement $a_1$ (Firm 2 abates ${A:.0f} - a_1$)")
    ax.set_ylabel("$ / unit")
    ax.set_xlim(0, a_max)
    y_top = max(slope1 * a_uniform * 1.3, slope2 * a_uniform * 1.3, 2)
    ax.set_ylim(0, y_top)
    ax.legend(fontsize=8, loc="upper left")

    # -- Right panel: cost bar chart --
    ax2 = axes[1]
    ax2.set_title("Total abatement cost", fontsize=11, color=UQ_PURPLE)
    bars = ax2.bar(["Uniform\nstandard", "Equimarginal\n(tax or trade)"],
                   [cost_un, cost_eq],
                   color=[UQ_PURPLE, UQ_CYAN], width=0.5, edgecolor="white", lw=1.5)
    for bar, val in zip(bars, [cost_un, cost_eq]):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"${val:.1f}", ha="center", va="bottom", fontsize=10,
                 fontweight="bold", color=UQ_PURPLE)
    if excess > 0.01:
        pct = 100 * excess / cost_un
        ax2.annotate(f"Excess: ${excess:.1f}\n({pct:.0f}%)",
                     xy=(1, cost_eq), xytext=(1.35, (cost_un + cost_eq) / 2),
                     fontsize=9, color=UQ_PURPLE, ha="center", fontweight="bold",
                     arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1.2))
    ax2.set_ylabel("Total cost ($)")
    ax2.set_ylim(0, max(cost_un * 1.5, 1))

    plt.tight_layout()
    plt.show()

interact(
    plot_uniform_standard,
    total_abatement=FloatSlider(value=10, min=2, max=20, step=0.5,
                               description="Total abatement:",
                               style={"description_width": "initial"}),
    slope1=FloatSlider(value=3.0, min=0.5, max=6.0, step=0.1,
                       description="MAC₁ slope (steep):",
                       style={"description_width": "initial"}),
    slope2=FloatSlider(value=1.0, min=0.5, max=6.0, step=0.1,
                       description="MAC₂ slope (flat):",
                       style={"description_width": "initial"}),
);

### Things to try
1. **Set both slopes equal**: the excess cost vanishes — identical firms means uniform = equimarginal.
2. **Widen the gap** between slopes: the excess cost triangle grows.
3. **Increase total abatement**: both costs rise, but the gap widens too.

---
## 2. Price instrument (tax) — cost-effective abatement

Each firm abates to $MAC_i = t$. Cheap firms abate more, expensive firms abate less.

- **Left panel**: Individual MAC curves with each firm's chosen abatement at the tax.
- **Right panel**: Total cost under the tax vs a uniform standard achieving the same total abatement.

In [ ]:
def make_slopes(n, heterogeneity):
    if n == 1:
        return np.array([2.0])
    base = 2.0
    spread = np.linspace(-heterogeneity, heterogeneity, n)
    return np.clip(base + spread, 0.3, 10.0)


def plot_tax(tax_rate=8.0, firms=4, heterogeneity=1.5):
    """Tax simulation: each firm abates to MAC_i = t."""
    n = int(firms)
    slopes = make_slopes(n, heterogeneity)
    t = tax_rate

    # Each firm: MAC_i(a_i) = slope_i * a_i => a_i = t / slope_i
    a_tax = np.minimum(t / slopes, E0_PER_FIRM)  # can't abate more than E0
    cost_tax = 0.5 * slopes * a_tax**2

    total_abatement = np.sum(a_tax)
    a_uniform = min(total_abatement / n, E0_PER_FIRM)
    cost_uniform = 0.5 * slopes * a_uniform**2

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Emission Tax: Cost-Effective Abatement",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    # -- Left: individual firms --
    ax1 = axes[0]
    ax1.set_title("Individual firms", fontsize=11, color=UQ_PURPLE)
    a_max = E0_PER_FIRM * 1.1
    a_range = np.linspace(0, a_max, 200)

    for i in range(n):
        c = COLORS[i % len(COLORS)]
        ax1.plot(a_range, slopes[i] * a_range, color=c, lw=1.5, alpha=0.7,
                 label=f"Firm {i+1}")
        # Tax-optimal point
        ax1.plot(a_tax[i], slopes[i] * a_tax[i], "o", color=c, ms=7, zorder=5)
        # Uniform point
        ax1.plot(a_uniform, slopes[i] * a_uniform, "x", color=c, ms=7, mew=2, zorder=5)

    ax1.axhline(t, color=UQ_PURPLE, ls="--", lw=1.5, alpha=0.6,
                label=f"Tax $t$ = {t:.1f}")
    ax1.set_xlim(0, a_max)
    ax1.set_ylim(0, max(t * 2.5, slopes[-1] * a_uniform * 1.5, 5))
    ax1.set_xlabel("Abatement $a_i$")
    ax1.set_ylabel("$ / unit")
    ax1.legend(fontsize=7, loc="upper left", framealpha=0.8)

    # -- Right: cost comparison --
    ax2 = axes[1]
    ax2.set_title("Total abatement cost", fontsize=11, color=UQ_PURPLE)
    tc_tax = np.sum(cost_tax)
    tc_uniform = np.sum(cost_uniform)
    saving = tc_uniform - tc_tax
    saving_pct = 100 * saving / tc_uniform if tc_uniform > 0 else 0

    bars = ax2.bar(["Uniform\nstandard", "Tax"],
                   [tc_uniform, tc_tax],
                   color=[UQ_PURPLE, UQ_CYAN], width=0.5, edgecolor="white", lw=1.5)
    for bar, val in zip(bars, [tc_uniform, tc_tax]):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"${val:.0f}", ha="center", va="bottom", fontsize=10,
                 fontweight="bold", color=UQ_PURPLE)
    if saving > 0.01:
        ax2.annotate(f"Saving: ${saving:.0f}\n({saving_pct:.0f}%)",
                     xy=(1, tc_tax), xytext=(1.35, (tc_uniform + tc_tax) / 2),
                     fontsize=9, color=UQ_PURPLE, ha="center", fontweight="bold",
                     arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1.2))
    ax2.set_ylabel("Total cost ($)")
    ax2.set_ylim(0, max(tc_uniform * 1.5, 1))

    plt.tight_layout()
    plt.show()


interact(
    plot_tax,
    tax_rate=FloatSlider(value=8.0, min=0.5, max=20.0, step=0.5,
                         description="Tax rate $t$:",
                         style={"description_width": "initial"}),
    firms=IntSlider(value=4, min=2, max=8, step=1,
                    description="Number of firms:",
                    style={"description_width": "initial"}),
    heterogeneity=FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1,
                              description="MAC heterogeneity:",
                              style={"description_width": "initial"}),
);

### Things to try
1. **Set heterogeneity to 0**: all firms are identical — tax and uniform standard give the same cost.
2. **Increase heterogeneity**: the circles (tax) spread apart while the crosses (uniform) stay bunched — the cost gap widens.
3. **Raise the tax**: more abatement from all firms, but the cheap ones still do the heavy lifting.

---
## 3. Cap-and-trade — the permit market

- **Left panel**: Individual firm MAC curves. Circles = post-trade; Crosses = uniform standard.
- **Centre panel**: Aggregate MAC meets the cap (vertical) — the permit price emerges.
- **Right panel**: Total abatement cost — trading vs uniform standard.

In [ ]:
def solve_trading(slopes, cap):
    n = len(slopes)
    total_abatement = max(n * E0_PER_FIRM - cap, 0.0)
    inv_slopes = 1.0 / slopes
    permit_price = total_abatement / np.sum(inv_slopes)
    a_star = permit_price / slopes
    cost_trading = 0.5 * slopes * a_star**2
    return permit_price, a_star, cost_trading


def solve_uniform(slopes, cap):
    n = len(slopes)
    total_abatement = max(n * E0_PER_FIRM - cap, 0.0)
    a_uniform = total_abatement / n
    cost_uniform = 0.5 * slopes * a_uniform**2
    return a_uniform, cost_uniform


def plot_cap_and_trade(cap=24, firms=4, heterogeneity=1.5):
    n = int(firms)
    slopes = make_slopes(n, heterogeneity)
    permit_price, a_star, cost_trading = solve_trading(slopes, cap)
    a_uniform, cost_uniform = solve_uniform(slopes, cap)
    total_E0 = n * E0_PER_FIRM

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Cap-and-Trade Simulation", fontsize=13,
                 fontweight="bold", color=UQ_PURPLE, y=1.02)
    plt.subplots_adjust(wspace=0.32)

    # -- Panel 1: Individual firms --
    ax1 = axes[0]
    ax1.set_title("Individual firms", fontsize=11, color=UQ_PURPLE)
    ax1.set_xlabel("Abatement $a_i$")
    ax1.set_ylabel("$ / unit")
    a_max = E0_PER_FIRM * 1.1
    a_range = np.linspace(0, a_max, 200)
    for i in range(n):
        c = COLORS[i % len(COLORS)]
        ax1.plot(a_range, slopes[i] * a_range, color=c, lw=1.5, alpha=0.7,
                 label=f"Firm {i+1}")
        ax1.plot(a_star[i], permit_price, "o", color=c, ms=7, zorder=5)
        ax1.plot(a_uniform, slopes[i] * a_uniform, "x", color=c, ms=7, mew=2, zorder=5)
    ax1.axhline(permit_price, color=UQ_PURPLE, ls="--", lw=1, alpha=0.6,
                label=f"$p$ = {permit_price:.1f}")
    ax1.set_xlim(0, a_max)
    ax1.set_ylim(0, max(permit_price * 2.5, 5))
    ax1.legend(fontsize=7, loc="upper left", framealpha=0.8)

    # -- Panel 2: Permit market --
    ax2 = axes[1]
    ax2.set_title("Permit market", fontsize=11, color=UQ_PURPLE)
    ax2.set_xlabel("Emissions $E$")
    ax2.set_ylabel("$ / unit")
    inv_sum = np.sum(1.0 / slopes)
    E_range = np.linspace(0, total_E0, 300)
    agg_mac = np.maximum((total_E0 - E_range) / inv_sum, 0)
    ax2.plot(E_range, agg_mac, color=UQ_PURPLE, lw=2.5, label="Aggregate MAC")
    ax2.axvline(cap, color=UQ_CYAN, lw=2.5, label=f"Cap = {cap:.0f}")
    ax2.plot([0, cap], [permit_price, permit_price], "--", color=UQ_PURPLE, lw=1, alpha=0.6)
    ax2.plot(cap, permit_price, "o", color=UQ_PURPLE, ms=8, zorder=5)
    ax2.annotate(f"$p$ = {permit_price:.1f}", xy=(cap, permit_price),
                 xytext=(cap + total_E0 * 0.05, permit_price + 0.5),
                 fontsize=9, color=UQ_PURPLE,
                 arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1))
    ax2.set_xlim(0, total_E0 * 1.05)
    ax2.set_ylim(0, max(agg_mac[0] * 1.2, 5))
    ax2.legend(fontsize=8, loc="upper right", framealpha=0.8)

    # -- Panel 3: Cost comparison --
    ax3 = axes[2]
    ax3.set_title("Total abatement cost", fontsize=11, color=UQ_PURPLE)
    tc_trade = np.sum(cost_trading)
    tc_uniform = np.sum(cost_uniform)
    saving = tc_uniform - tc_trade
    saving_pct = 100 * saving / tc_uniform if tc_uniform > 0 else 0
    bars = ax3.bar(["Uniform\nstandard", "Cap-and-\ntrade"],
                   [tc_uniform, tc_trade],
                   color=[UQ_PURPLE, UQ_CYAN], width=0.5, edgecolor="white", lw=1.5)
    if saving > 0.01:
        ax3.annotate(f"Saving: ${saving:.0f}\n({saving_pct:.0f}%)",
                     xy=(1, tc_trade), xytext=(1.35, (tc_uniform + tc_trade) / 2),
                     fontsize=9, color=UQ_PURPLE, ha="center", fontweight="bold",
                     arrowprops=dict(arrowstyle="->", color=UQ_PURPLE, lw=1.2))
    ax3.set_ylabel("Total cost ($)")
    ax3.set_ylim(0, max(tc_uniform * 1.4, 1))
    for bar, val in zip(bars, [tc_uniform, tc_trade]):
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"${val:.0f}", ha="center", va="bottom", fontsize=9,
                 fontweight="bold", color=UQ_PURPLE)

    plt.tight_layout()
    plt.show()


interact(
    plot_cap_and_trade,
    cap=IntSlider(value=24, min=5, max=75, step=1,
                  description="Cap (total E):",
                  style={"description_width": "initial"}),
    firms=IntSlider(value=4, min=2, max=8, step=1,
                    description="Number of firms:",
                    style={"description_width": "initial"}),
    heterogeneity=FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1,
                              description="MAC heterogeneity:",
                              style={"description_width": "initial"}),
);

### Things to try
1. **Tighten the cap** (slide left): permit price rises, costs increase — but trading always beats the uniform standard.
2. **Set heterogeneity to 0**: all firms are identical — trading saves nothing.
3. **Increase heterogeneity**: the cost gap widens — more diverse firms = bigger gains from trade.
4. **Set cap = total E₀** (n × 10): no abatement needed — price = 0, costs = 0.

---
## 4. Weitzman (1974) — prices vs quantities under uncertainty

The regulator sets a **tax** $t$ or a **cap** $\bar{q}$ using the *expected* MAC. Then a cost shock $\theta$ is realised.

- **Left panel**: Quantity instrument — the cap is stuck at $\bar{q}$ while the true optimum shifts. The deadweight loss (DWL) triangle depends on $b + c$.
- **Right panel**: Price instrument — firms adjust quantity to the shock, but overshoot/undershoot the true optimum. The DWL depends on $b^2 / c^2$.

The **Weitzman rule**: use a price when MAC is steeper than MD ($c > b$), use a quantity when MD is steeper ($b > c$).

In [ ]:
def plot_weitzman(b_slope=1.0, c_slope=2.0, theta=3.0):
    """
    Weitzman prices vs quantities.
    MB(q) = beta - b*q  (marginal benefit = marginal damage avoided)
    MC(q; theta) = gamma + c*q + theta  (marginal abatement cost)
    """
    b, c = b_slope, c_slope
    beta, gamma = 10.0, 1.0  # intercepts

    # Expected optimum (theta=0)
    q_bar = (beta - gamma) / (b + c)
    t_bar = gamma + c * q_bar  # = beta - b * q_bar

    # True optimum
    q_star = (beta - gamma - theta) / (b + c)

    # Price instrument: firms set MC = t_bar => gamma + c*q + theta = t_bar
    q_price = max((t_bar - gamma - theta) / c, 0)

    # Quantity instrument: q stays at q_bar
    q_quant = q_bar

    # For plotting
    q_max = max(q_bar * 2, q_star * 1.5, 3) if q_star > 0 else q_bar * 2
    q_range = np.linspace(0, q_max, 300)

    mb = beta - b * q_range
    mc_exp = gamma + c * q_range
    mc_true = gamma + c * q_range + theta

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    fig.suptitle("Weitzman (1974): Prices vs Quantities under Uncertainty",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE, y=1.02)

    titles = ["Quantity instrument (cap)", "Price instrument (tax)"]
    q_instruments = [q_quant, q_price]

    for idx, ax in enumerate(axes):
        ax.set_title(titles[idx], fontsize=11, color=UQ_PURPLE)
        ax.set_xlabel("Abatement $q$")
        ax.set_ylabel("$ / unit")

        # MB curve
        ax.plot(q_range, mb, color=UQ_PURPLE, lw=2, label="$MB$ (marginal benefit)")
        # Expected MC
        ax.plot(q_range, mc_exp, color=UQ_CYAN, lw=1.5, ls="--", alpha=0.5,
                label="$MC^e$ (expected)")
        # True MC
        ax.plot(q_range, mc_true, color=UQ_CYAN, lw=2, label="$MC$ (realised)")

        # True optimum
        if q_star > 0:
            y_star = beta - b * q_star
            ax.plot(q_star, y_star, "o", color="gray", ms=7, zorder=5, label=f"$q^*={q_star:.1f}$")
            ax.axvline(q_star, color="gray", ls=":", lw=1, alpha=0.4)

        # Instrument outcome
        q_inst = q_instruments[idx]
        if q_inst > 0 and q_star > 0:
            # DWL triangle between q_inst and q_star
            q_lo = min(q_inst, q_star)
            q_hi = max(q_inst, q_star)
            q_fill = np.linspace(q_lo, q_hi, 100)
            mb_fill = beta - b * q_fill
            mc_fill = gamma + c * q_fill + theta
            ax.fill_between(q_fill, mb_fill, mc_fill, alpha=0.3, color="#E6308A",
                            label="DWL")

        if idx == 0:
            # Quantity: show the cap
            ax.axvline(q_bar, color=UQ_CYAN, lw=2.5, alpha=0.8)
            ax.text(q_bar + 0.1, ax.get_ylim()[1] * 0.1 if ax.get_ylim()[1] > 0 else 1,
                    f"cap $\\bar q={q_bar:.1f}$", fontsize=9, color=UQ_CYAN,
                    rotation=90, va="bottom")
        else:
            # Price: show the tax line
            ax.axhline(t_bar, color=UQ_PURPLE, lw=2, ls="--", alpha=0.6)
            ax.text(0.1, t_bar + 0.2, f"tax $t={t_bar:.1f}$", fontsize=9,
                    color=UQ_PURPLE)
            if q_price > 0:
                ax.axvline(q_price, color=UQ_CYAN, lw=1.5, ls="--", alpha=0.5)
                ax.text(q_price + 0.1, 0.5, f"$q_p={q_price:.1f}$", fontsize=8,
                        color=UQ_CYAN)

        ax.set_xlim(0, q_max)
        y_top = max(beta * 1.1, (gamma + c * q_max + abs(theta)) * 1.05)
        ax.set_ylim(0, y_top)
        ax.legend(fontsize=7, loc="upper right", framealpha=0.8)

    # Summary statistics
    sigma2 = theta**2  # single realisation
    loss_Q = theta**2 / (2 * (b + c))
    loss_P = b**2 * theta**2 / (2 * c**2 * (b + c))
    delta = sigma2 / (2 * c**2) * (c - b)

    winner = "PRICE" if delta > 0 else ("QUANTITY" if delta < 0 else "TIE")
    fig.text(0.5, -0.02,
             f"DWL(quantity) = {loss_Q:.2f}     DWL(price) = {loss_P:.2f}     "
             f"$\\Delta$ = {delta:+.2f}  →  {winner} wins",
             ha="center", fontsize=10, color=UQ_PURPLE, fontweight="bold")

    plt.tight_layout()
    plt.show()


interact(
    plot_weitzman,
    b_slope=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                        description="MD slope ($b$):",
                        style={"description_width": "initial"}),
    c_slope=FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1,
                        description="MAC slope ($c$):",
                        style={"description_width": "initial"}),
    theta=FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.5,
                      description="Cost shock ($\\theta$):",
                      style={"description_width": "initial"}),
);

### Things to try
1. **Set $b > c$** (steep MD, flat MAC): the quantity DWL shrinks, price DWL grows — **quantities win**. Think toxic pollutants with a critical threshold.
2. **Set $c > b$** (steep MAC, flat MD): the price DWL shrinks — **prices win**. Think CO₂ where annual marginal damage is nearly flat.
3. **Set $b = c$**: it's a tie — $\Delta = 0$.
4. **Increase $|\theta|$**: both DWL triangles grow, but the *ratio* stays the same — it's the slopes that determine the winner, not the shock size.
5. **Set $\theta = 0$**: no shock — both instruments hit the optimum perfectly, DWL = 0.